# Brownian Diffusion
We here show how to set up an Analysis object and use it to first fit an artificial vanadium measurement to obtain the resolution. Next, we use the fitted resolution to fit an artificial measurement of a model with diffusion and some elastic scattering. 

We extract and plot the relevant parameters. Finally, we show how to fit directly to the diffusion model.

In the near future, it will be possible to fit the width and area of the Lorentzian to the diffusion model as well.

In [ ]:
# Imports
import pooch

from easydynamics.analysis.analysis import Analysis
from easydynamics.experiment import Experiment
from easydynamics.sample_model import BrownianTranslationalDiffusion
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import DeltaFunction
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model import Lorentzian
from easydynamics.sample_model import Polynomial
from easydynamics.sample_model.background_model import BackgroundModel
from easydynamics.sample_model.instrument_model import InstrumentModel
from easydynamics.sample_model.resolution_model import ResolutionModel
from easydynamics.sample_model.sample_model import SampleModel
import scipp as sc
import numpy as np

# Make the plots interactive
%matplotlib widget

We first create an `Experiment` object to contain the data. The data must either be a hdf5 file or a scipp.DataArray; in both cases it must have coordinates `Q` and `energy`. We here use Pooch to download an example vanadium data set.

In [ ]:
resolution_experiment = Experiment(display_name = 'Nanoparticles, 1.5 K', data='data/nano_1p5K.h5')

In [ ]:
resolution_experiment.plot_data(slicer = True, keep = 'energy')

In [ ]:
# Limit the energy range to the region around the elastic line
emin=-0.2*sc.Unit('meV')
emax=0.2*sc.Unit('meV')
resolution_experiment.data = resolution_experiment.data['energy',emin:emax]

resolution_experiment.plot_data(slicer = True, keep = 'energy')

In [ ]:
delta_function = DeltaFunction(area=100)
res_sample_model = SampleModel(components=delta_function)

res_resolution_model = ResolutionModel()
res_components = ComponentCollection()
res_gauss = Gaussian(area=1, width =0.02)
res_gauss.area.fixed = True

res_components.append_component(res_gauss)
res_resolution_model.components = res_components

background_model = BackgroundModel()
polynomial = Polynomial(coefficients=[1.5])
polynomial.coefficients[0].min = 0.0
background_model.components = polynomial


res_instrument_model = InstrumentModel(resolution_model=res_resolution_model, background_model=background_model)

res_analysis = Analysis(experiment=resolution_experiment, sample_model=res_sample_model, instrument_model=res_instrument_model)

res_analysis.plot_data_and_model( keep = 'energy')

In [ ]:
# The fit is not perfect, but it's pretty good. Exercise for the reader: add more components?
res_analysis.fit()
res_analysis.plot_data_and_model( keep = 'energy')


In [ ]:
experiment = Experiment(display_name = 'Nanoparticles, 150 K', data='data/nano_150K.h5')

emin=-1.5*sc.Unit('meV')
emax=1.5*sc.Unit('meV')
experiment.data = experiment.data['energy',emin:emax]
experiment.data.variances[~np.isfinite(experiment.data.values)] = 1.0
experiment.data.values[~np.isfinite(experiment.data.values)] = 0.0

experiment.plot_data(slicer = True, keep = 'energy')

In [ ]:
sample_model = SampleModel()
water_delta_function = DeltaFunction(display_name = 'Water delta function',area=100)
water_lorentzian = Lorentzian(display_name = 'Water Lorentzian', area=100, width=0.2)
sample_model.append_component(water_delta_function)
sample_model.append_component(water_lorentzian)
sample_model.temperature = 150


background_model = BackgroundModel()
polynomial = Polynomial(coefficients=[1.5, 0.0])
background_model.components = polynomial


instrument_model = InstrumentModel(background_model=background_model)


analysis = Analysis(experiment=experiment, sample_model=sample_model, instrument_model=instrument_model)
analysis.instrument_model._resolution_model = (
    res_analysis.instrument_model.resolution_model
)
analysis.instrument_model.resolution_model.fix_all_parameters()

analysis.plot_data_and_model( keep = 'energy')

In [ ]:
analysis.fit()
analysis.plot_data_and_model( keep = 'energy')

In [ ]:
# Inspect background parameters - they look constant for the non-magnetic Q
analysis.plot_parameters(names = ['Water delta function area', 'Water Lorentzian area'])

In [ ]:
analysis.plot_parameters(names = ['Water Lorentzian width'])

In [ ]:
# Look at components
delta_0 = analysis.sample_model.get_component_collection(Q_index=0).components[0]
delta_1 = analysis.sample_model.get_component_collection(Q_index=1).components[0]
delta_area = (delta_0.area+delta_1.area)/2


lorz_0 = analysis.sample_model.get_component_collection(Q_index=0).components[1]
lorz_1 = analysis.sample_model.get_component_collection(Q_index=1).components[1]
lorz_area = (lorz_0.area+lorz_1.area)/2
lorz_width = (lorz_0.width+lorz_1.width)/2


for Q_index in range(analysis.sample_model.Q.size):
    delta = analysis.sample_model.get_component_collection(Q_index=Q_index).components[0]
    delta.area = delta_area.value
    delta.area.fixed = True

    lorz = analysis.sample_model.get_component_collection(Q_index=Q_index).components[1]
    lorz.area = lorz_area.value
    lorz.width = lorz_width.value
    lorz.area.fixed = True
    lorz.width.fixed = True

analysis.plot_data_and_model( keep = 'energy')